In [ ]:
!pip install diffusers transformers accelerate
!pip install moviepy
!pip install gtts
!pip install pillow
!apt-get install -y imagemagick
!pip install scipy

In [ ]:
import os
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image, ImageDraw, ImageFont
import numpy as np
from moviepy.editor import (
    ImageClip, concatenate_videoclips, CompositeVideoClip,
    AudioFileClip, TextClip, CompositeAudioClip
)
from gtts import gTTS
import json
import re
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Configuration
class Config:
    OUTPUT_DIR = "/content/book_videos"
    TEMP_DIR = "/content/temp"
    VIDEO_WIDTH = 1080
    VIDEO_HEIGHT = 1920
    FPS = 24
    TOTAL_DURATION = 60  # 1 minute
    SCENE_DURATION = 5
    IMAGE_STEPS = 25
    MUSIC_VOLUME = 0.25

os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
os.makedirs(Config.TEMP_DIR, exist_ok=True)

In [ ]:
GENRE_STYLES = {
    "fantasy": {
        "visual": "epic fantasy landscape, magical, ethereal, cinematic lighting, detailed",
        "colors": "purple and blue mystical tones"
    },
    "sci-fi": {
        "visual": "futuristic, cyberpunk aesthetic, neon lights, high-tech, space",
        "colors": "cyan and dark blue tech tones"
    },
    "romance": {
        "visual": "romantic, soft lighting, beautiful scenery, emotional, dreamy",
        "colors": "warm pink and gold tones"
    },
    "thriller": {
        "visual": "dark, mysterious, dramatic shadows, suspenseful, tense",
        "colors": "dark red and black tones"
    },
    "mystery": {
        "visual": "noir style, moody, atmospheric fog, mysterious",
        "colors": "dark blue and grey tones"
    },
    "horror": {
        "visual": "dark, eerie, ominous atmosphere, gothic",
        "colors": "blood red and pitch black"
    },
    "adventure": {
        "visual": "dynamic action, exciting, vibrant landscapes, epic journey",
        "colors": "bright and colorful adventure tones"
    },
    "literary": {
        "visual": "artistic, thoughtful, elegant composition, refined",
        "colors": "muted sophisticated tones"
    }
}

In [ ]:
def detect_genre(summary: str) -> str:
    """Auto-detect book genre from summary"""
    summary_lower = summary.lower()

    keywords = {
        "fantasy": ["magic", "wizard", "dragon", "fantasy", "kingdom", "quest", "spell"],
        "sci-fi": ["space", "future", "robot", "ai", "alien", "technology", "cyber", "planet"],
        "romance": ["love", "heart", "relationship", "romance", "passion", "wedding"],
        "thriller": ["danger", "chase", "escape", "threat", "suspense", "hunt"],
        "mystery": ["detective", "mystery", "clue", "investigation", "murder", "solve"],
        "horror": ["horror", "terror", "fear", "ghost", "monster", "haunted", "nightmare"],
        "adventure": ["journey", "adventure", "explore", "discovery", "voyage", "expedition"]
    }

    for genre, words in keywords.items():
        if any(word in summary_lower for word in words):
            return genre
    return "literary"

def split_into_scenes(summary: str, num_scenes: int = 8) -> List[str]:
    """Split summary into scene descriptions"""
    sentences = re.split(r'[.!?]+', summary)
    sentences = [s.strip() for s in sentences if s.strip()]

    if len(sentences) <= num_scenes:
        return sentences

    scenes_per_group = len(sentences) // num_scenes
    scenes = []

    for i in range(num_scenes):
        start = i * scenes_per_group
        end = start + scenes_per_group if i < num_scenes - 1 else len(sentences)
        scenes.append(' '.join(sentences[start:end]))

    return scenes

def create_prompts(summary: str, genre: str) -> List[Dict]:
    """Create image generation prompts"""
    scenes = split_into_scenes(summary, 6)
    style = GENRE_STYLES.get(genre, GENRE_STYLES["literary"])

    quality_boost = ", RAW photo, detailed face, detailed eyes, photorealistic, 8k uhd, highly detailed, professional photography"
    negative = "cartoon, 3d, blurry, deformed, ugly, bad anatomy, bad hands, distorted face, low quality"

    prompts = []

    # Title card
    prompts.append({
        "type": "title",
        "text": "A New Story Awaits",
        "prompt": f"elegant book cover design, {style['visual']}, {style['colors']}, professional, high quality",
        "negative": negative
    })

    # Scene images
    for scene in scenes:
        prompts.append({
            "type": "scene",
            "text": scene[:100],
            "prompt": f"{scene}, {style['visual']}, {style['colors']}, cinematic, highly detailed, masterpiece",
            "negative": negative
        })

    # Ending card
    prompts.append({
        "type": "ending",
        "text": "Read More...",
        "prompt": f"inspiring book finale, {style['visual']}, {style['colors']}, hopeful, beautiful",
        "negative": negative
    })

    return prompts

In [ ]:
Config.VIDEO_WIDTH = 608
Config.VIDEO_HEIGHT = 1080
Config.IMAGE_STEPS = 20

In [ ]:
print("Loading Stable Diffusion model (this may take a few minutes)...")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if device == "cpu":
    print("⚠️ WARNING: GPU not detected. Video generation will be VERY slow.")
    print("Please change runtime: Runtime > Change runtime type > GPU")

pipe = StableDiffusionPipeline.from_pretrained(
    "SG161222/Realistic_Vision_V5.1_noVAE",  # Better faces!
    torch_dtype=torch.float16,
    safety_checker=None
)
pipe = pipe.to(device)

if device == "cuda":
    pipe.enable_attention_slicing()
    pipe.enable_vae_slicing()

def generate_image(prompt: str, negative_prompt: str = "") -> Image.Image:
    """Generate image from text prompt"""
    width = (Config.VIDEO_WIDTH // 8) * 8
    height = (Config.VIDEO_HEIGHT // 8) * 8

    image = pipe(
        prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=Config.IMAGE_STEPS,
        guidance_scale=7.5,
        width=width,
        height=height
    ).images[0]

    return image

def add_text_to_image(image: Image.Image, text: str) -> Image.Image:
    """Add centered text overlay"""
    draw = ImageDraw.Draw(image)
    width, height = image.size

    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 80)
    except:
        font = ImageFont.load_default()

    bbox = draw.textbbox((0, 0), text, font=font)
    text_width = bbox[2] - bbox[0]
    text_height = bbox[3] - bbox[1]

    x = (width - text_width) / 2
    y = (height - text_height) / 2

    # Shadow
    draw.text((x + 3, y + 3), text, font=font, fill=(0, 0, 0, 200))
    # Text
    draw.text((x, y), text, font=font, fill=(255, 255, 255, 255))

    return image

In [ ]:
from scipy.io import wavfile

def create_narration(text: str, output_path: str):
    narration_text = f"Discover a captivating story. {text}"
    tts = gTTS(text=narration_text, lang='en', slow=False)
    tts.save(output_path)

    # Split into chunks for subtitle display (~4 words per chunk)
    words = narration_text.split()
    chunks = [' '.join(words[i:i+4]) for i in range(0, len(words), 4)]
    return chunks

def create_background_music(duration: float, output_path: str):
    """Generate simple ambient background music"""
    sample_rate = 44100
    t = np.linspace(0, duration, int(sample_rate * duration))

    # Ambient chord progression
    freqs = [220, 277, 330, 440]  # A minor 7th chord
    audio = np.zeros_like(t)

    for i, freq in enumerate(freqs):
        audio += np.sin(2 * np.pi * freq * t) * (0.25 / len(freqs))

    # Add subtle variation
    audio *= (1 + 0.2 * np.sin(2 * np.pi * 0.1 * t))

    # Fade in/out
    fade = int(sample_rate * 3)
    audio[:fade] *= np.linspace(0, 1, fade)
    audio[-fade:] *= np.linspace(1, 0, fade)

    # Normalize and convert
    audio = (audio / np.max(np.abs(audio)) * 0.4 * 32767).astype(np.int16)

    wavfile.write(output_path, sample_rate, audio)
    print(f"✓ Background music created: {output_path}")

In [ ]:
def make_subtitle_clip(text, duration, video_width, video_height):
    """Pillow-based subtitle — no ImageMagick needed"""
    from PIL import Image, ImageDraw, ImageFont
    import numpy as np

    img = Image.new("RGBA", (video_width, 120), (0, 0, 0, 0))
    draw = ImageDraw.Draw(img)

    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 45)
    except:
        font = ImageFont.load_default()

    # Center the text
    bbox = draw.textbbox((0, 0), text, font=font)
    text_w = bbox[2] - bbox[0]
    x = (video_width - text_w) / 2

    # Shadow then white text
    draw.text((x + 2, 42), text, font=font, fill=(0, 0, 0, 220))
    draw.text((x, 40), text, font=font, fill=(255, 255, 255, 255))

    frame = np.array(img)
    clip = (ImageClip(frame, ismask=False)
            .set_duration(duration)
            .set_position(('center', video_height - 180)))
    return clip


def compose_video(image_paths, narration_path, music_path, output_path, subtitle_chunks=None):
    print("Composing final video...")

    # Determine total duration from narration
    if os.path.exists(narration_path):
        narration_audio = AudioFileClip(narration_path)
        total_duration = narration_audio.duration + 1
    else:
        total_duration = Config.TOTAL_DURATION

    duration_per_image = total_duration / len(image_paths)

    # Build image clips
    clips = []
    for i, img_path in enumerate(image_paths):
        clip = ImageClip(img_path, duration=duration_per_image)
        clip = clip.resize(lambda t: 1 + 0.03 * t / duration_per_image)
        if i > 0:
            clip = clip.crossfadein(0.5)
        if i < len(image_paths) - 1:
            clip = clip.crossfadeout(0.5)
        clips.append(clip)

    video = concatenate_videoclips(clips, method="compose")

    # Build subtitle clips using Pillow (no ImageMagick)
    subtitle_clips = []
    if subtitle_chunks:
        chunk_duration = total_duration / len(subtitle_chunks)
        for i, chunk in enumerate(subtitle_chunks):
            txt_clip = (make_subtitle_clip(chunk, chunk_duration,
                                           Config.VIDEO_WIDTH, Config.VIDEO_HEIGHT)
                        .set_start(i * chunk_duration))
            subtitle_clips.append(txt_clip)

    # Composite subtitles over video
    if subtitle_clips:
        video = CompositeVideoClip(
            [video] + subtitle_clips,
            size=(Config.VIDEO_WIDTH, Config.VIDEO_HEIGHT)
        )

    # Audio
    audio_clips = []
    if os.path.exists(narration_path):
        audio_clips.append(narration_audio)

    if os.path.exists(music_path):
        music = AudioFileClip(music_path)
        music = music.volumex(Config.MUSIC_VOLUME)
        music = (music.audio_loop(duration=total_duration)
                 if music.duration < total_duration
                 else music.subclip(0, total_duration))
        audio_clips.append(music)

    if audio_clips:
        video = video.set_audio(CompositeAudioClip(audio_clips))

    video.write_videofile(
        output_path,
        fps=Config.FPS,
        codec='libx264',
        audio_codec='aac',
        threads=4,
        preset='medium'
    )
    print(f"✓ Video saved: {output_path}")

In [ ]:
!apt-get update -qq
!apt-get install -y imagemagick
!which convert  # Should print /usr/bin/convert

In [ ]:
!sed -i 's/rights="none" pattern="PDF"/rights="read|write" pattern="PDF"/' /etc/ImageMagick-6/policy.xml

from moviepy.config import change_settings
import subprocess

# Find wherever convert actually lives
result = subprocess.run(["which", "convert"], capture_output=True, text=True)
convert_path = result.stdout.strip()
print(f"Found convert at: {convert_path}")

change_settings({"IMAGEMAGICK_BINARY": convert_path})

In [ ]:
def generate_book_video(summary: str, title: str = "Untitled"):
    """
    Main function to generate video from book summary

    Args:
        summary: Book summary text
        title: Book title (used for filename)

    Returns:
        Path to generated video
    """
    print(f"\n{'='*60}")
    print(f"Generating video for: {title}")
    print(f"{'='*60}\n")

    # Detect genre
    genre = detect_genre(summary)
    print(f"📚 Detected genre: {genre.upper()}")

    # Create prompts
    prompts = create_prompts(summary, genre)
    print(f"📝 Created {len(prompts)} scene prompts")

    # Generate images
    print(f"\n🎨 Generating images (this will take several minutes)...")
    image_dir = os.path.join(Config.TEMP_DIR, "images")
    os.makedirs(image_dir, exist_ok=True)

    image_paths = []
    for i, prompt_data in enumerate(prompts):
        print(f"  Generating image {i+1}/{len(prompts)}: {prompt_data['type']}")

        # Generate with negative prompts for better quality
        image = generate_image(
            prompt_data['prompt'],
            prompt_data.get('negative', '')
        )

        # Add text for title/ending
        if prompt_data['type'] in ['title', 'ending']:
            image = add_text_to_image(image, prompt_data['text'])

        path = os.path.join(image_dir, f"scene_{i:03d}.png")
        image.save(path)
        image_paths.append(path)

    # Generate audio
    print(f"\n🎙️ Creating narration and music...")
    narration_path = os.path.join(Config.TEMP_DIR, "narration.mp3")
    music_path = os.path.join(Config.TEMP_DIR, "music.wav")

    subtitle_chunks = create_narration(summary, narration_path)  # now returns chunks

    # Get real duration for music
    narration_audio = AudioFileClip(narration_path)
    actual_duration = narration_audio.duration + 1
    narration_audio.close()

    create_background_music(actual_duration, music_path)

    print(f"\n🎬 Composing final video...")
    output_path = os.path.join(Config.OUTPUT_DIR, f"{title.replace(' ', '_')}_reel.mp4")
    compose_video(image_paths, narration_path, music_path, output_path,
                  subtitle_chunks=subtitle_chunks)

    print(f"\n{'='*60}")
    print(f"✅ SUCCESS! Video generated!")
    print(f"📁 Location: {output_path}")
    print(f"{'='*60}\n")

    return output_path

In [ ]:
BOOK_TITLE = "The Forgotten Magic"

BOOK_SUMMARY = """
In a world where magic has been forgotten, a young orphan named Elena discovers
an ancient artifact that awakens her dormant powers. As darkness threatens to
consume the kingdom, she must embark on a perilous journey to unite the scattered
magical societies. Along the way, she encounters mysterious allies, faces powerful
enemies, and uncovers the truth about her own heritage. The fate of the realm
rests in her hands as she learns that true strength comes not from power, but
from the bonds we forge and the courage to stand against impossible odds.
Will she master her abilities in time to save her world?
"""

In [ ]:
video_path = generate_book_video(BOOK_SUMMARY, BOOK_TITLE)

# Display the video in Colab
from IPython.display import Video
Video(video_path, width=800)

In [ ]:
from google.colab import files

# Download the video
files.download(video_path)

print(f"✓ Video downloaded: {os.path.basename(video_path)}")

In [ ]:
import os
print(os.getcwd())
print(os.listdir())